# Titanic Survival Prediction — My ML Pipeline

This is my walkthrough of the Titanic dataset for the Kaggle "Titanic — Machine Learning from Disaster" competition. I've laid it out as the full workflow I use for a tabular classification problem, with my own notes on why I made each choice along the way.

**My pipeline:**
1. Data Collection
2. Exploratory Data Analysis (EDA) + visualizations
3. Data Cleaning & Preprocessing (missing values)
4. Feature Engineering
5. Feature Extraction
6. Feature Selection
7. Encoding
8. Scaling
9. Train/Test Split
10. Cross-Validation (CV)
11. Models (multiple algorithms)
12. Ensembling
13. Prediction
14. Accuracy
15. Confusion Matrix
16. Kaggle Submission


## 1. Data Collection

I start by loading the raw data — nothing else is possible until it's in the notebook. Kaggle gives me two CSV files for this competition:
- `train.csv` — 891 passengers, includes `Survived` (the label I'm trying to predict)
- `test.csv` — 418 passengers, no `Survived` column (this is what I predict for my submission)

The cell below auto-detects whether I'm running on Kaggle (`/kaggle/input/titanic/`) or locally, so I don't have to edit paths depending on where I run it.

In [ ]:
import os
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

# Auto-detect the Kaggle input path for this competition, fall back to local files
KAGGLE_DIR = "/kaggle/input/titanic"
DATA_DIR = KAGGLE_DIR if os.path.exists(KAGGLE_DIR) else "."

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()

I keep the test passenger IDs aside right now — I'll need them at the very end to build my submission files, and it's easy to lose track of them once I start reshaping the data.

In [ ]:
test_passenger_ids = test["PassengerId"].copy()

## 2. Exploratory Data Analysis (EDA)

Before I touch anything, I like to look at the data first — what columns exist, what type of data they hold, how much is missing, and how features relate to my target (`Survived`). This is the stage that guides every decision I make later, like how to clean the data and what features to build. Skipping it is the easiest way to end up with a weak model.

### 2.1 Basic structure

In [ ]:
train.info()

In [ ]:
train.describe()  # summary stats for numeric columns: count, mean, std, min, max, quartiles

### 2.2 Target variable balance

Before I model anything, I want to check if `Survived` is balanced (roughly 50/50) or imbalanced, since an imbalanced target changes how I should evaluate a model — accuracy alone can be misleading if 90% of passengers belonged to one class.

In [ ]:
print(train["Survived"].value_counts(normalize=True))

plt.figure(figsize=(5,4))
sns.countplot(data=train, x="Survived")
plt.title("Survived Count (0 = Died, 1 = Survived)")
plt.show()

**What I'm seeing:** Titanic is mildly imbalanced (~62% died, ~38% survived) — not extreme, so I'll treat plain accuracy as a reasonable metric here, but I'll still check the confusion matrix later to make sure my model isn't just guessing the majority class.

### 2.3 Missing values overview

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

plt.figure(figsize=(6,4))
sns.barplot(x=missing.values, y=missing.index)
plt.title("Missing Values per Column (train)")
plt.xlabel("Count missing")
plt.show()

**What I'm seeing:** `Cabin` is missing for most passengers, `Age` for about 1 in 5, and `Embarked` for just 2. This tells me exactly which columns need attention in my cleaning stage, and roughly how aggressive each fix needs to be.

### 2.4 Survival by Sex

"Women and children first" — I want to check if the data actually supports that.

In [ ]:
plt.figure(figsize=(5,4))
sns.barplot(data=train, x="Sex", y="Survived")
plt.title("Survival Rate by Sex")
plt.ylabel("Survival Rate")
plt.show()

print(train.groupby("Sex")["Survived"].mean())

**What I'm seeing:** women survived at a much higher rate than men. This makes `Sex` the single strongest predictor in the whole dataset — I expect every model I try to lean on it heavily.

### 2.5 Survival by Passenger Class (Pclass)

Pclass (1st/2nd/3rd) is a proxy for wealth, and wealth correlated with cabin location (closer to lifeboats) and priority during evacuation.

In [ ]:
plt.figure(figsize=(5,4))
sns.barplot(data=train, x="Pclass", y="Survived")
plt.title("Survival Rate by Passenger Class")
plt.ylabel("Survival Rate")
plt.show()

print(train.groupby("Pclass")["Survived"].mean())

### 2.6 Age distribution, split by survival

A histogram lets me see whether certain age ranges (young children, for example) survived at noticeably different rates.

In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(data=train, x="Age", hue="Survived", bins=30, kde=True, multiple="stack")
plt.title("Age Distribution by Survival")
plt.show()

**What I'm seeing:** young children (roughly under 10) show a visibly higher survival bump — consistent with them being prioritized for lifeboats. This tells me binning Age (rather than using it raw) might help my models pick up on this pattern more easily.

### 2.7 Fare distribution, split by survival

In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(data=train, x="Fare", hue="Survived", bins=40)
plt.title("Fare Distribution by Survival")
plt.xlim(0, 300)
plt.show()

**What I'm seeing:** higher fares (correlated with 1st class) skew toward survival, which reinforces what I already saw with Pclass.

### 2.8 Correlation heatmap (numeric columns only)

A heatmap shows me which numeric columns move together. It's a quick way for me to spot redundant features (highly correlated columns often carry the same information) and to see which numeric features relate most to `Survived`.

In [ ]:
numeric_df = train.select_dtypes(include=[np.number]).drop(columns=["PassengerId"])
corr = numeric_df.corr()

plt.figure(figsize=(7,6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap (numeric columns)")
plt.show()

**What I'm seeing:** `Fare` has the strongest positive correlation with `Survived` among the raw numeric columns; `Pclass` has a negative correlation with `Survived` (lower class number = higher class = more likely to survive, so Pclass 3 correlates with dying). Nothing here is so highly correlated with another feature that I need to drop it outright — but I'll revisit this properly in my Feature Selection stage, after I've engineered new features.

## 3. Data Cleaning & Preprocessing

Now I fix missing values, inconsistent formats, and anything obviously wrong before feeding the data to a model — models can't handle missing values (NaN) directly, so this step isn't optional for me.

I combine train and test into one `full` DataFrame here so every cleaning/engineering step I apply lands on both identically — this avoids a common bug where train and test end up with mismatched columns.

In [ ]:
train["is_train"] = 1
test["is_train"] = 0
test["Survived"] = np.nan  # test has no label; placeholder keeps columns aligned

full = pd.concat([train, test], sort=False).reset_index(drop=True)
print(full.shape)

### 3.1 `Age` — fill using group medians, not one global number

Filling every missing Age with the overall average throws away real differences between groups (a young "Master" vs. an adult "Mr" have very different typical ages). So I first extract a `Title` from the passenger's name (Mr, Mrs, Miss, Master, or Rare), then fill missing Age using the **median Age within each Title + Pclass group** — a much more accurate guess than a single global number.

In [ ]:
def extract_title(name: str) -> str:
    match = re.search(r",\s*([^\.]*)\.", name)
    return match.group(1).strip() if match else "Unknown"

full["Title"] = full["Name"].apply(extract_title)

title_map = {
    "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
    "Lady": "Rare", "Countess": "Rare", "Capt": "Rare", "Col": "Rare",
    "Don": "Rare", "Dr": "Rare", "Major": "Rare", "Rev": "Rare",
    "Sir": "Rare", "Jonkheer": "Rare", "Dona": "Rare"
}
full["Title"] = full["Title"].replace(title_map)
full["Title"] = full["Title"].where(full["Title"].isin(["Mr", "Mrs", "Miss", "Master", "Rare"]), "Rare")

full["Age"] = full.groupby(["Title", "Pclass"])["Age"].transform(lambda x: x.fillna(x.median()))
full["Age"] = full["Age"].fillna(full["Age"].median())  # safety net for any leftover NaNs

print("Age missing after fill:", full["Age"].isnull().sum())

### 3.2 `Cabin` — too sparse to impute, so I extract what I can

77% of `Cabin` is missing — there's no reliable way for me to guess a specific cabin. But the missingness itself is informative (poorer passengers' cabins often weren't recorded), so instead of dropping the column, I extract the **Deck letter** and label missing ones `"Unknown"` as their own category rather than imputing a fake value.

In [ ]:
full["Deck"] = full["Cabin"].apply(lambda x: x[0] if pd.notnull(x) else "Unknown")

deck_counts = full["Deck"].value_counts()
rare_decks = deck_counts[deck_counts < 15].index
full["Deck"] = full["Deck"].replace(rare_decks, "Rare")

print(full["Deck"].value_counts())

### 3.3 `Embarked` — only 2 missing, fill with the mode

With just 2 missing values out of 1309 total rows, I'll fill with the most common port ("S" = Southampton) — it's a safe, low-risk choice, since the effect on the model is negligible either way.

In [ ]:
full["Embarked"] = full["Embarked"].fillna(full["Embarked"].mode()[0])
print("Embarked missing after fill:", full["Embarked"].isnull().sum())

### 3.4 `Fare` — 1 missing (test set), fill using Pclass median

Filling with the overall fare average would be skewed upward by expensive 1st-class fares, so I use the median fare **within that passenger's Pclass** instead — it gives a much closer real-world estimate.

In [ ]:
full["Fare"] = full.groupby("Pclass")["Fare"].transform(lambda x: x.fillna(x.median()))
print("Fare missing after fill:", full["Fare"].isnull().sum())

### 3.5 Final check — no missing values left (except `Survived` for test rows, which is expected)

In [ ]:
remaining = full.drop(columns=["Survived"]).isnull().sum()
print(remaining[remaining > 0] if (remaining > 0).any() else "All clean.")

## 4. Feature Engineering

Here I create brand-new columns from existing ones to give my model more useful signal than the raw data alone provides. This is usually where I see the biggest accuracy gains — more than from model choice.

### 4.1 Family size features

`SibSp` (siblings/spouses) and `Parch` (parents/children) are more useful to me combined. Survival wasn't linear with family size — both solo travelers and very large families fared worse than small families — so I bucket it into categories instead of leaving it as a raw number.

In [ ]:
full["FamilySize"] = full["SibSp"] + full["Parch"] + 1

def family_bucket(size):
    if size == 1:
        return "Alone"
    elif size <= 4:
        return "Small"
    else:
        return "Large"

full["FamilyBucket"] = full["FamilySize"].apply(family_bucket)
full["IsAlone"] = (full["FamilySize"] == 1).astype(int)

plt.figure(figsize=(5,4))
sns.barplot(data=full[full["is_train"]==1], x="FamilyBucket", y="Survived", order=["Alone","Small","Large"])
plt.title("Survival Rate by Family Size Bucket")
plt.show()

### 4.2 Age and Fare binning

Binning turns a continuous number into categories, which can help tree-based models find cleaner splits on a small dataset, and helps linear models capture non-linear patterns — like the child-survival bump I saw in my EDA.

In [ ]:
full["AgeBin"] = pd.cut(full["Age"], bins=[0, 12, 18, 35, 60, 100],
                         labels=["Child", "Teen", "Adult", "Middle", "Senior"])

full["FarePerPerson"] = full["Fare"] / full["FamilySize"]
full["FareBin"] = pd.qcut(full["FarePerPerson"], 4, labels=["Low", "Mid", "High", "VeryHigh"])

full[["Age","AgeBin","Fare","FarePerPerson","FareBin"]].head()

### 4.3 Ticket group size

Passengers sharing an identical ticket number often traveled together (friends, servants) even if `SibSp`/`Parch` don't count them as family — a signal those columns miss on their own, so I add it separately.

In [ ]:
ticket_counts = full["Ticket"].value_counts()
full["TicketGroupSize"] = full["Ticket"].map(ticket_counts)
full[["Ticket","TicketGroupSize"]].head()

## 5. Feature Extraction

**How I think about this V/s ature Engineering:** Feature Engineering (above) *combines or transforms* existing numeric/categorical columns (family size, bins). Feature **Extraction** is me pulling brand-new information out of unstructured or semi-structured fields — text, in this case — that wasn't a usable column at all in the raw data.

I already did the main example of this: `Title` extracted from the free-text `Name` field, and `Deck` extracted from the `Cabin` string. Both took a messy text field and gave me a clean, structured signal my model can use. Let me confirm what I extracted and check its relationship with survival, since I haven't visualized it yet.

In [ ]:
plt.figure(figsize=(6,4))
sns.barplot(data=full[full["is_train"]==1], x="Title", y="Survived",
            order=full["Title"].value_counts().index)
plt.title("Survival Rate by Extracted Title")
plt.show()

**What I'm seeing:** `Title` extraction turned out to be extremely valuable — survival rate varies a lot by title even within the same Sex/Pclass, which is exactly why pulling it out of free text was worth the effort.

## 6. Feature Selection

Now I decide which features actually help my model and drop the ones that add noise, redundancy, or no signal. More features isn't always better — irrelevant or redundant ones can hurt performance and slow training.

I do this two ways:
1. **Manual/logical selection** — I drop columns I've already mined for signal (raw `Name`, `Ticket`, `Cabin` text) and pure identifiers (`PassengerId`) that have no predictive meaning on their own.
2. **Data-driven selection** — I train a quick Random Forest and look at its feature importance scores, to confirm my engineered features are actually pulling weight (this needs encoding first, so I do a light version here and revisit after full encoding in Section 7).

In [ ]:
# Manual selection: drop columns that are either raw text already mined, or pure IDs
drop_cols = ["PassengerId", "Name", "Ticket", "Cabin", "is_train"]

print("Columns being dropped (already mined or non-predictive):", drop_cols)
print("\nRemaining columns going into modeling:")
print([c for c in full.columns if c not in drop_cols + ["Survived"]])

## 7. Encoding

Models only understand numbers, so I convert text categories (like "male"/"female" or "S"/"C"/"Q") into numeric columns. I use **one-hot encoding**, which creates a separate 0/1 column for each category — e.g. `Sex_male` becomes 1 if male, 0 if female. I drop one category per column (`drop_first=True`) to avoid redundant, perfectly-correlated columns — if I know a passenger isn't "female", and there are only 2 sexes, I already know they're "male", so keeping both columns would be redundant.

In [ ]:
categorical_cols = ["Sex", "Embarked", "Title", "FamilyBucket", "AgeBin", "FareBin", "Deck"]
full_encoded = pd.get_dummies(full, columns=categorical_cols, drop_first=True)

feature_cols = [c for c in full_encoded.columns if c not in
                ["PassengerId", "Survived", "Name", "Ticket", "Cabin", "is_train"]]

print("Total features after encoding:", len(feature_cols))
full_encoded[feature_cols].head()

### 7.1 Data-driven feature selection with Random Forest importance

Now that everything is numeric, I can train a quick Random Forest and check which features it actually relies on. Features with near-zero importance would be candidates to drop in a more advanced pass — I'm keeping all of them for this pipeline, since Titanic has few enough features that pruning isn't critical, but this is exactly how I'd decide what to cut on a larger dataset.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X_check = full_encoded[full_encoded["is_train"] == 1][feature_cols]
y_check = full_encoded[full_encoded["is_train"] == 1]["Survived"].astype(int)

quick_rf = RandomForestClassifier(n_estimators=300, random_state=42)
quick_rf.fit(X_check, y_check)

importances = pd.Series(quick_rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(7,8))
sns.barplot(x=importances.values[:15], y=importances.index[:15])
plt.title("Top 15 Feature Importances (Random Forest)")
plt.show()

**What I'm seeing:** `Fare`, `Age`, `Sex_male`, and the extracted `Title` columns typically rank highest — which confirms my feature engineering and extraction steps were worth doing, since several top features didn't exist in the raw data.

## 8. Scaling

Here I rescale numeric columns so they're on comparable ranges. `Age` (0-80) and `Fare` (0-500+) are on very different scales — this doesn't matter for tree-based models (Random Forest, XGBoost split on raw values regardless of scale), **but it matters a lot for Logistic Regression and any distance-based model (KNN, SVM)**, since those algorithms treat larger numeric ranges as more "important" purely due to scale, not actual signal.

I use `StandardScaler`, which transforms each column to have mean 0 and standard deviation 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

X_full = full_encoded[full_encoded["is_train"] == 1][feature_cols].reset_index(drop=True)
y_full = full_encoded[full_encoded["is_train"] == 1]["Survived"].astype(int).reset_index(drop=True)
X_test_final = full_encoded[full_encoded["is_train"] == 0][feature_cols].reset_index(drop=True)

numeric_to_scale = ["Age", "Fare", "FamilySize", "FarePerPerson", "TicketGroupSize"]

scaler = StandardScaler()
X_full_scaled = X_full.copy()
X_test_scaled = X_test_final.copy()

X_full_scaled[numeric_to_scale] = scaler.fit_transform(X_full[numeric_to_scale])
X_test_scaled[numeric_to_scale] = scaler.transform(X_test_final[numeric_to_scale])

X_full_scaled[numeric_to_scale].describe()

**A rule I stick to:** I `fit_transform` on the training data only, then `transform` (not `fit_transform` again) on the test data, using the *same* scaler. This avoids "data leakage" — if I fit the scaler on test data too, information from the test set would subtly leak into training, giving me an overly optimistic (and dishonest) sense of model performance.

## 9. Train/Test Split

I set aside part of my labeled data purely to check how well my model performs on data it hasn't seen — simulating the real Kaggle test set, which I can't check myself before submitting. I use `train_test_split` to carve out 20% of `train.csv` as my local validation set. `stratify=y_full` keeps both the training and validation splits at the same Survived ratio, which matters since my target is mildly imbalanced, as I saw in EDA.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_full_scaled, y_full, test_size=0.2, random_state=42, stratify=y_full
)

print("Training rows:", X_train.shape[0])
print("Validation rows:", X_val.shape[0])

## 10. Cross-Validation (CV)

Instead of trusting just one 80/20 split (which could be lucky or unlucky), I repeat the split multiple times in different ways and average the results. **Stratified 5-fold CV** splits my training data into 5 equal parts, trains on 4 and validates on 1, five times (rotating which part is held out), then averages the 5 accuracy scores. This gives me a far more trustworthy estimate of how a model will generalize than a single split — and it's what I use to actually compare and choose models.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 11. Models

I train and compare several classic algorithms, each with my own one-line note on how it thinks about the problem:

- **Logistic Regression** — draws a straight-line-style decision boundary; fast and interpretable.
- **K-Nearest Neighbors (KNN)** — classifies a passenger based on the majority vote of the most "similar" passengers (by feature distance). This is why I scaled the data — KNN is a distance-based model.
- **Decision Tree** — asks a series of yes/no questions (e.g. "Sex_male == 1?", "Age < 12?") to split passengers into groups.
- **Random Forest** — trains many Decision Trees on random subsets and averages their votes; usually much stronger than one tree alone.
- **Gradient Boosting** — builds trees one at a time, each new tree specifically correcting the mistakes of the previous ones.

I evaluate all of them the same way — 5-fold CV accuracy — for a fair comparison.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42),
}

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
    cv_results[name] = scores
    print(f"{name:20s} CV Accuracy: {scores.mean():.4f}  (+/- {scores.std():.4f})")

In [ ]:
plt.figure(figsize=(8,5))
plt.boxplot(cv_results.values(), labels=cv_results.keys())
plt.title("CV Accuracy Distribution by Model")
plt.ylabel("Accuracy")
plt.xticks(rotation=20)
plt.show()

**What I'm seeing:** the boxplot shows not just the average CV accuracy per model, but how much it varies across folds — a model with a high average but wide spread is less reliable to me than one with a slightly lower average but tight, consistent scores.

## 12. Ensembling

Here I combine multiple models' predictions into one, on the idea that different models make different mistakes, so averaging them out can improve overall accuracy. I use a **soft-voting classifier**, which averages each model's predicted *probability* of survival (rather than just their hard 0/1 vote) across the models I pick, then makes a final decision from that average.

In [ ]:
from sklearn.ensemble import VotingClassifier

ensemble = VotingClassifier(
    estimators=[
        ("lr", models["Logistic Regression"]),
        ("rf", models["Random Forest"]),
        ("gb", models["Gradient Boosting"]),
    ],
    voting="soft"
)

ensemble_scores = cross_val_score(ensemble, X_train, y_train, cv=cv, scoring="accuracy")
print(f"Ensemble CV Accuracy: {ensemble_scores.mean():.4f}  (+/- {ensemble_scores.std():.4f})")

## 13. Final Training, Prediction & Accuracy

I pick whichever performed best in CV — single model or ensemble — fit it on my *full* training split (`X_train`, not the CV folds), and evaluate it on the validation set I held out back in Section 9. This validation accuracy is my best honest estimate of how the model will do on brand-new data, like Kaggle's actual test set.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

best_name = max(cv_results, key=lambda k: cv_results[k].mean())
best_cv_score = cv_results[best_name].mean()

if ensemble_scores.mean() >= best_cv_score:
    final_model = ensemble
    print("Using ENSEMBLE as the final model")
else:
    final_model = models[best_name]
    print(f"Using {best_name} as the final model")

final_model.fit(X_train, y_train)
val_predictions = final_model.predict(X_val)

val_accuracy = accuracy_score(y_val, val_predictions)
print(f"\nValidation Accuracy: {val_accuracy:.4f}")

## 14. Confusion Matrix

**What this is:** a table comparing actual outcomes vs. predicted outcomes, broken into 4 categories:
- **True Negative (TN)** — actually died, correctly predicted died
- **False Positive (FP)** — actually died, incorrectly predicted survived
- **False Negative (FN)** — actually survived, incorrectly predicted died
- **True Positive (TP)** — actually survived, correctly predicted survived

This matters to me because accuracy alone can hide a model that's, say, great at predicting deaths but bad at predicting survivals (or vice versa) — especially on an imbalanced target. The confusion matrix exposes exactly where my model's mistakes are concentrated.

In [ ]:
cm = confusion_matrix(y_val, val_predictions)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted: Died","Predicted: Survived"],
            yticklabels=["Actual: Died","Actual: Survived"])
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_val, val_predictions, target_names=["Died","Survived"]))

**How I read the classification report:**
- **Precision** — of everyone the model predicted as "Survived", what fraction actually survived (measures false-alarm rate)
- **Recall** — of everyone who actually survived, what fraction the model correctly caught (measures miss rate)
- **F1-score** — a balance between precision and recall in one number
- **Support** — how many actual examples of each class were in the validation set

## 15. Generate the Kaggle Submission File

Finally, I retrain my final model on **all** available training data (not just the 80% split — I want to use every labeled row I have before predicting on the real, unlabeled test set), then predict on the actual Kaggle test set and write the submission file in the exact format Kaggle requires: `PassengerId`, `Survived`.

I also save it to `/kaggle/working/submission.csv` when I'm running inside a Kaggle notebook, since that's the path Kaggle looks at for competition submissions.

In [ ]:
final_model.fit(X_full_scaled, y_full)
test_predictions = final_model.predict(X_test_scaled).astype(int)

submission = pd.DataFrame({
    "PassengerId": test_passenger_ids,
    "Survived": test_predictions
})

# Write to /kaggle/working when running on Kaggle, otherwise the local folder
OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
submission_path = f"{OUTPUT_DIR}/submission.csv"
submission.to_csv(submission_path, index=False)

print(f"submission.csv written to {submission_path} with {len(submission)} rows.")
submission.head()

## Summary — my full pipeline, in order

| # | Stage | What I did | Key tool/function |
|---|---|---|---|
| 1 | Data Collection | Loaded train/test CSVs (Kaggle-aware path) | `pd.read_csv` |
| 2 | EDA + Visuals | Checked shape, missing values, survival by Sex/Pclass/Age/Fare, correlation heatmap | `seaborn`, `matplotlib`, `.groupby()` |
| 3 | Cleaning/Preprocessing | Filled Age (group median), Cabin→Deck, Embarked (mode), Fare (Pclass median) | `.fillna()`, `.transform()` |
| 4 | Feature Engineering | FamilySize, FamilyBucket, AgeBin, FareBin, TicketGroupSize | `pd.cut`, `pd.qcut` |
| 5 | Feature Extraction | Title from Name, Deck from Cabin (raw text → structured signal) | `re.search`, `.apply()` |
| 6 | Feature Selection | Dropped IDs/raw text; checked Random Forest importances | `RandomForestClassifier.feature_importances_` |
| 7 | Encoding | One-hot encoded categorical columns | `pd.get_dummies` |
| 8 | Scaling | Standardized numeric columns (mean 0, std 1) | `StandardScaler` |
| 9 | Train/Test Split | Held out 20% for honest validation | `train_test_split` |
| 10 | Cross-Validation | 5-fold stratified CV for reliable model comparison | `StratifiedKFold`, `cross_val_score` |
| 11 | Models | Logistic Regression, KNN, Decision Tree, Random Forest, Gradient Boosting | sklearn classifiers |
| 12 | Ensembling | Soft-voting across top 3 models | `VotingClassifier` |
| 13 | Prediction & Accuracy | Fit best model, predicted on held-out validation set | `.predict()`, `accuracy_score` |
| 14 | Confusion Matrix | Visualized where the model succeeds/fails | `confusion_matrix`, `classification_report` |
| 15 | Submission | Retrained on full data, predicted on real Kaggle test set, saved to `/kaggle/working/` | `.to_csv()` |

**My biggest takeaway:** the largest accuracy gains in this whole pipeline came from Steps 3–5 (cleaning, engineering, extraction) — not from which model I picked in Step 11. That pattern holds true on almost every real-world tabular dataset I've worked with, not just Titanic.